In [ ]:
# Training: Loss and Stochastic Gradient Descent
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part1/04-training-loss-sgd.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part1').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part1')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Generate one least-squares problem and solve for its empirical optimum.
2. Partition the examples into equal batches and define their gradients.
3. Compare batch descent directions far from and at the optimum.
4. Verify that their fixed-point mean equals the full descent direction, then plot both zones.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# [1]
torch.manual_seed(6050)
x1 = torch.randn(80)
y1 = 2.5 * x1 - 1.0 + 0.4 * torch.randn(80)
A = torch.stack([x1, torch.ones_like(x1)], dim=1)
optimum = torch.linalg.lstsq(A, y1).solution

# [2]
batch_size = 4
batches = torch.randperm(len(x1)).reshape(-1, batch_size)

def gradient_at(theta: torch.Tensor, idx=slice(None)) -> torch.Tensor:
    residual = A[idx] @ theta - y1[idx]
    return 2 * A[idx].T @ residual / len(residual)

def batch_descent_directions(theta: torch.Tensor) -> torch.Tensor:
    return torch.stack([-gradient_at(theta, idx) for idx in batches])

# [3]
points = [torch.tensor([-0.5, 2.0]), optimum]
titles = ["far away: agreement in expectation",
          "at the optimum: batches disagree"]
arrow_lengths = [0.72, 0.46]

W, B = np.meshgrid(np.linspace(-1.5, 5.5, 100),
                   np.linspace(-4.0, 3.0, 100))
L = ((y1.numpy()[None, None, :] -
      (W[..., None] * x1.numpy() + B[..., None])) ** 2).mean(-1)

# [4]
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7), sharex=True, sharey=True)
for ax, theta, title, length in zip(axes, points, titles, arrow_lengths):
    directions = batch_descent_directions(theta)
    full_direction = -gradient_at(theta)
    assert torch.allclose(directions.mean(0), full_direction, atol=2e-6)

    unit = directions / directions.norm(dim=1, keepdim=True).clamp_min(1e-12)
    origin = theta.repeat(len(unit), 1)
    ax.contour(W, B, L, levels=24, cmap="Blues", alpha=0.55)
    ax.quiver(origin[:, 0], origin[:, 1], unit[:, 0], unit[:, 1],
              color="#E57200", alpha=0.32, angles="xy",
              scale_units="xy", scale=1 / length, width=0.008)

    if full_direction.norm() > 1e-5:
        mean_unit = full_direction / full_direction.norm()
        end = theta + 1.15 * mean_unit
        ax.annotate("", xy=end, xytext=theta,
                    arrowprops={"arrowstyle": "->", "lw": 3, "color": "#232D4B"})
    else:
        ax.text(float(theta[0]), float(theta[1]) + 0.72,
                "mean direction $\\approx 0$", ha="center", color="#232D4B")

    ax.plot(*optimum, marker="*", ms=13, color="#2F855A")
    ax.plot(*theta, marker="o", ms=5, color="#232D4B")
    ax.set(title=title, xlabel="$w$", xlim=(-1.5, 5.5), ylim=(-4, 3))
    ax.grid(alpha=0.12)

axes[0].set_ylabel("$b$")
axes[0].plot([], [], color="#E57200", lw=3, alpha=0.45,
             label="batch directions")
axes[0].plot([], [], color="#232D4B", lw=3,
             label="mean = full direction")
axes[0].plot([], [], marker="*", ms=10, color="#2F855A", lw=0,
             label="empirical optimum")
axes[0].legend(loc="upper right", fontsize=8)
axes[1].add_patch(plt.Circle(optimum.numpy(), 0.82, color="#722F37",
                             alpha=0.09, zorder=0))
axes[1].text(float(optimum[0]), float(optimum[1]) - 1.05,
             "region of confusion", ha="center", color="#722F37")
plt.tight_layout()
plt.show()

**Plan**

1. Define the reusable `losses_for` helper.
2. Prepare the inputs and fixed settings for the example.
3. Implement the learning-rate triptych.
4. Report or visualize the measured result.

In [ ]:
# [1]
def losses_for(lr: float, steps: int = 60) -> list[float]:
    w, b, out = torch.tensor(-0.5), torch.tensor(2.0), []
    for _ in range(steps):
        err = (w * x1 + b) - y1
        out.append(float((err ** 2).mean()))
        w = w - lr * 2 * (err * x1).mean()
        b = b - lr * 2 * err.mean()
    return out

# [2]
plt.figure(figsize=(6.2, 3.4))
# [3]
for lr, style, color in [(0.005, "-", "#5379AA"), (0.12, "-", "#E57200"),
                         (1.1, "--", "#722F37")]:
    plt.semilogy(losses_for(lr), style, color=color, label=f"$\\alpha = {lr}$")
plt.xlabel("step"); plt.ylabel("loss (log scale)")
# [4]
plt.legend(); plt.tight_layout(); plt.show()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable `race` helper.
3. Implement the optimizer race.
4. Report or visualize the measured result.

In [ ]:
# [1]
torch.manual_seed(6050)
n = 256
Xr = torch.randn(n, 2) * torch.tensor([1.0, 10.0])   # ill-conditioned by design
w_true = torch.tensor([3.0, 0.3])
yr = Xr @ w_true + 0.1 * torch.randn(n)

# [2]
def race(kind: str, steps: int = 120, B: int = 16) -> list[float]:
    torch.manual_seed(1)                             # same batches for everyone
    w = torch.zeros(2, requires_grad=True)
    opt = {"SGD":      torch.optim.SGD([w], lr=3e-3),
           "momentum": torch.optim.SGD([w], lr=3e-3, momentum=0.9),
           "Adam":     torch.optim.Adam([w], lr=0.1)}[kind]
    hist = []
    for _ in range(steps):
        idx = torch.randint(0, n, (B,))
        opt.zero_grad()
        ((Xr[idx] @ w - yr[idx]) ** 2).mean().backward()
        opt.step()
        hist.append(((Xr @ w.detach() - yr) ** 2).mean().item())
    return hist

plt.figure(figsize=(6.2, 3.6))
# [3]
for kind, color in [("SGD", "#5379AA"), ("momentum", "#232D4B"),
                    ("Adam", "#E57200")]:
    plt.semilogy(race(kind), color=color, label=kind)
plt.xlabel("step"); plt.ylabel("full-batch loss (log scale)")
# [4]
plt.legend(); plt.tight_layout(); plt.show()

**Plan**

1. Define the reusable `inverted_dropout` helper.
2. Prepare the inputs and fixed settings for the example.
3. Implement inverted dropout and verify its scale.

In [ ]:
# [1]
def inverted_dropout(
    h: torch.Tensor, p: float, training: bool = True
) -> torch.Tensor:
    if not training:
        return h
    if not 0.0 <= p < 1.0:
        raise ValueError("p must satisfy 0 <= p < 1")
    q = 1.0 - p
    mask = (torch.rand_like(h) < q).to(h.dtype)
    return mask * h / q

# [2]
torch.manual_seed(6050)
h = torch.tensor([1.0, -2.0, 4.0, 0.5])
draws = torch.stack([inverted_dropout(h, p=0.25) for _ in range(20_000)])
mean_output = draws.mean(0)
# [3]
print("training mean:", [round(v, 4) for v in mean_output.tolist()])
print(f"max |mean - h|: {(mean_output - h).abs().max():.4f}")
print("evaluation:", inverted_dropout(h, p=0.25, training=False).tolist())

**Plan**

1. Declare one reusable interface for the model factory, data, and training budget.
2. Seed before constructing the model and optimizer.
3. Reshuffle the examples and visit every minibatch each epoch.
4. Predict, measure cross-entropy, backpropagate, and update.
5. Return the trained model as the experiment artifact.

In [ ]:
"""Listing 4.1 — the supervised training loop, importable.

Chapter 4 derives this loop and prints it; Chapters 6, 8, and 9 import it and
print only their deltas (the model builder and the budget). The loop is the
book's canonical minibatch recipe: seed, build, then repeat
predict -> measure -> step over reshuffled minibatches.
"""
from collections.abc import Callable

import torch
import torch.nn.functional as F
from torch import nn


# [1]
def fit_supervised(
    model_fn: Callable[[], nn.Module],
    X: torch.Tensor,
    y: torch.Tensor,
    *,
    epochs: int,
    batch: int = 64,
    lr: float = 1e-3,
    seed: int = 6050,
) -> nn.Module:
    """Train a fresh model on (X, y) with cross-entropy and Adam.

    Seeding precedes construction, so a given (model_fn, seed) pair always
    starts from the same tensors; each epoch reshuffles example order.
    """
    # [2]
    torch.manual_seed(seed)
    model = model_fn()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # [3]
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), batch):
            idx = perm[i : i + batch]
            # [4]
            loss = F.cross_entropy(model(X[idx]), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    # [5]
    return model